# 03. Level of detection and significant change

Block 02 left a DoD, `dod_debiased`, whose stable ground is centered near zero with an NMAD of about 8 cm. In this block we turn that noise into a level of detection (LoD), the smallest change we are willing to call real, and see how the choice of LoD changes the volumes.

In [ ]:
import csv

from grass.tools import Tools

from workshop_utils import gj, rounded, stable_stats, start

tools, session = start(
    "03", required=["dod_debiased", "stable", "slope_2013", "burned"]
)
# A second Tools instance that keeps message output, for tools that report numbers there.
report = Tools(capture_stderr=True, consistent_return_value=True)
n_region = tools.g_region(flags="p", format="json")["cells"]

## A single LoD

*r.dem.lod* `method=global` estimates the NMAD on stable ground and multiplies it by the normal critical value for the confidence level, 1.96 at 95%.

In [ ]:
result = report.r_dem_lod(
    dod="dod_debiased",
    output="lod_global",
    method="global",
    stable_mask="stable",
    confidence=0.95,
)
print(result.stderr.strip())

## A local LoD needs stable ground nearby

`method=local` estimates the NMAD in a moving window instead. With a stable mask, it only defines the LoD where the window reaches stable cells, and it does not extend it beyond them. Our stable ground is the unburned corner of the region.

In [ ]:
result = report.r_dem_lod(
    dod="dod_debiased",
    output="lod_local",
    method="local",
    stable_mask="stable",
    window=21,
)
print(result.stderr.strip().splitlines()[0])

A local LoD covers about 15% of the region and almost none of the burn scar, so it cannot test the change we care about here. It works when stable ground is spread through the study area.

## An LoD from terrain

The NMAD grows with slope (block 01), and slope is known everywhere. Estimate the NMAD of the stable residuals by slope class, then assign each cell the NMAD of its slope class. The error model is fit on stable ground and carried to the burn scar by slope, not by distance.

In [ ]:
edges = [0, 10, 20, 30, 40, 90]
sigma_by_class = []
print(f"{'slope (deg)':>12}{'median':>10}{'NMAD':>10}{'cells':>10}")
for lo, hi in zip(edges[:-1], edges[1:]):
    tools.r_mapcalc(
        expression=f"slope_bin = if(slope_2013 >= {lo} && slope_2013 < {hi}, 1, null())"
    )
    tools.r_mapcalc(
        expression="stable_bin = if(!isnull(stable) && !isnull(slope_bin), 1, null())"
    )
    stats = stable_stats(tools, "dod_debiased", "stable_bin")
    sigma_by_class.append(stats["nmad"])
    print(
        f"{f'{lo}-{hi}':>12}{stats['median']:>+10.3f}{stats['nmad']:>10.3f}{stats['n']:>10}"
    )

# Nested if() assigns each cell the NMAD of its slope class.
expression = "sigma_slope = "
for hi, sigma in zip(edges[1:], sigma_by_class):
    expression += f"if(slope_2013 < {hi}, {sigma}, "
expression += "null()" + ")" * len(sigma_by_class)
tools.r_mapcalc(expression=expression)
tools.r_mapcalc(expression="lod_slope = 1.96 * sigma_slope")

In [ ]:
lod_map = gj.Map(width=700)
lod_map.d_rast(map="lod_slope")
lod_map.d_vect(
    map="mckinney_perimeter@PERMANENT", fill_color="none", color="black", width=2
)
lod_map.d_legend(raster="lod_slope", title="LoD (m)", at=(5, 40, 2, 5), flags="b")
lod_map.show()

## Significant change and volumes

*r.dem.change* keeps the cells whose change exceeds the LoD, removes isolated significant cells with `-n`, and writes the volumes. It calls lowering erosion and raising deposition.

In [ ]:
def volumes(lod, name):
    tools.r_dem_change(
        dod="dod_debiased",
        lod=lod,
        output_sig=f"sig_{name}",
        volume_csv=f"volumes_{name}.csv",
        flags="n",
    )
    with open(f"volumes_{name}.csv") as f:
        return {row["metric"]: float(row["value_m3"]) for row in csv.DictReader(f)}


results = {
    name: volumes(lod, name)
    for name, lod in (("global", "lod_global"), ("slope", "lod_slope"))
}

# For comparison, the volumes with no LoD at all (1 m cells, so m3 = m * 1 m2).
tools.r_mapcalc(expression="dod_lowering = if(dod_debiased < 0, dod_debiased, null())")
tools.r_mapcalc(expression="dod_raising = if(dod_debiased > 0, dod_debiased, null())")
results["no LoD"] = {
    "erosion": -tools.r_univar(map="dod_lowering", format="json")["sum"],
    "deposition": tools.r_univar(map="dod_raising", format="json")["sum"],
}
results["no LoD"]["net"] = (
    results["no LoD"]["deposition"] - results["no LoD"]["erosion"]
)

print(f"{'':10}{'erosion m3':>14}{'deposition m3':>16}{'net m3':>14}")
for name, v in results.items():
    print(
        f"{name:10}{rounded(v['erosion']):>14,.0f}{rounded(v['deposition']):>16,.0f}"
        f"{rounded(v['net']):>+14,.0f}"
    )

About two-thirds of the lowering and 60% of the raising survive either LoD, because much of the change on the burn scar is larger than 15 cm. Surface lowering on a burn scar is not all eroded sediment, so treat these as volumes of surface change. They are printed to two significant figures. A leftover bias of 4 mm on stable ground is about 26,000 m3 over the region, so the last digits carry no information. Block 04 looks at where they happen.

## Check the LoD against stable ground

At 95% confidence, about 5% of stable cells should come out significant by chance. Count them.

In [ ]:
for name, lod in (("global", "lod_global"), ("slope", "lod_slope")):
    tools.r_mapcalc(
        expression=f"flagged = if(!isnull(stable) && !isnull(sig_{name}), 1, null())"
    )
    flagged = tools.r_univar(map="flagged", format="json")["n"]
    tested = stable_stats(tools, lod, "stable")["n"]
    print(
        f"{name:7}: {100 * flagged / tested:.1f}% of stable cells flagged significant"
    )

About twice the nominal 5%. The stable residuals have heavier tails than a normal distribution, so an NMAD-based 95% limit is optimistic here. A higher confidence level, or an LoD calibrated so that 5% of stable cells pass, is the honest response. The same applies to the class labels below. They state confidence from the normal model, so read "Erosion >=99%" as strong evidence of lowering, not a literal 99%.

## Significance classes

*r.dem.errprop* turns the slope-based uncertainty into z-scores and classes of confidence for erosion and deposition. Its color table runs from red for confident erosion through white to blue for confident deposition.

In [ ]:
tools.r_dem_errprop(
    dod="dod_debiased",
    sigma="sigma_slope",
    output_sigma="sigma_total",
    output_class="change_class",
    confidence=0.95,
)
print(tools.r_stats(input="change_class", flags="lcn").text)

In [ ]:
classes = gj.Map(width=700)
classes.d_shade(shade="dtm_2022_relief", color="change_class")
classes.d_vect(
    map="mckinney_perimeter@PERMANENT", fill_color="none", color="black", width=2
)
classes.d_legend(
    raster="change_class",
    title="Change class",
    at=(50, 92, 2, 5),
    fontsize=11,
    flags="b",
)
classes.show()

## Your turn

Rerun the slope-based LoD at 68%, 95%, and 99% confidence and watch the trade between the volume you keep and the share of stable cells that pass by chance.

In [ ]:
z_values = {"68%": 1.0, "95%": 1.96, "99%": 2.576}
print(f"{'level':>6}{'erosion m3':>14}{'deposition m3':>16}{'stable flagged':>16}")
for level, z in z_values.items():
    tools.r_mapcalc(expression=f"lod_try = {z} * sigma_slope")
    v = volumes("lod_try", "try")
    tools.r_mapcalc(
        expression="flagged = if(!isnull(stable) && !isnull(sig_try), 1, null())"
    )
    flagged = tools.r_univar(map="flagged", format="json")["n"]
    tested = stable_stats(tools, "lod_try", "stable")["n"]
    print(
        f"{level:>6}{rounded(v['erosion']):>14,.0f}{rounded(v['deposition']):>16,.0f}"
        f"{100 * flagged / tested:>15.1f}%"
    )